# 🚀 GenAI Restoration Studio - Google Colab T4 Bootstrap & Setup

This bootstrap notebook initializes the Google Colab T4 GPU environment, connects to Google Drive for persistent datasets/checkpoints/MLflow logs, pulls the latest project code, and verifies the data pipelines and PyTorch dataset modules.

### Step 1: Mount Google Drive & Configure Persistent Paths

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define persistent paths in Google Drive
DRIVE_ROOT = '/content/drive/MyDrive/GenAI-A1'
RAW_OXFORD_DIR = os.path.join(DRIVE_ROOT, 'raw', 'OxfordPet')
RAW_FS2K_DIR = os.path.join(DRIVE_ROOT, 'raw', 'FS2K')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
MLRUNS_DIR = os.path.join(DRIVE_ROOT, 'mlruns')
MANIFESTS_DIR = os.path.join(DRIVE_ROOT, 'manifests')
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')

for p in [DRIVE_ROOT, RAW_OXFORD_DIR, RAW_FS2K_DIR, CHECKPOINTS_DIR, MLRUNS_DIR, MANIFESTS_DIR, EXPORTS_DIR]:
    os.makedirs(p, exist_ok=True)

print("✅ Google Drive mounted and directories initialized:")
print(f"  Persistent Root: {DRIVE_ROOT}")
print(f"  Checkpoints:     {CHECKPOINTS_DIR}")
print(f"  MLflow Logs:     {MLRUNS_DIR}")

### Step 2: Clone or Update Project Repository

In [ ]:
import sys

REPO_URL = 'https://github.com/UsmanBari/genai-restoration-studio.git'
REPO_DIR = '/content/genai-restoration-studio'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("✅ Working directory set to repo:", os.getcwd())

### Step 3: Install Training Dependencies & Check CUDA GPU

In [ ]:
!pip install -q -r requirements-colab.txt

import torch
import torchvision
import optuna
import mlflow
import onnx
import onnxruntime

print("✅ PyTorch version:", torch.__version__)
print("✅ CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("✅ GPU Device:", torch.cuda.get_device_name(0))
    print("✅ Memory Allocated:", round(torch.cuda.memory_allocated(0)/(1024**3), 2), "GB")
else:
    print("⚠️ Warning: Running on CPU. Switch Colab runtime to T4 GPU (Runtime > Change runtime type).")

### Step 4: Prepare Oxford-IIIT Pet Dataset (Run Once)

In [ ]:
from scripts.prepare_oxford_pet import prepare_oxford_pet

print("Preparing Oxford-IIIT Pet Dataset on Google Drive...")
prepare_oxford_pet(
    output_dir=RAW_OXFORD_DIR,
    manifest_dir=MANIFESTS_DIR,
    target_size=(128, 128),
    split_seed=42
)
print("✅ Oxford-IIIT Pet Dataset ready!")

### Step 5: Unpack & Prepare FS2K Dataset (Run Once)

In [ ]:
from scripts.prepare_fs2k import prepare_fs2k

print("Unpacking and verifying FS2K Dataset on Google Drive...")
prepare_fs2k(
    fs2k_dir=RAW_FS2K_DIR,
    manifest_dir=MANIFESTS_DIR,
    split_seed=42
)
print("✅ FS2K Dataset ready!")

### Step 6: Verify PyTorch DataLoaders & Corruption Pipeline

In [ ]:
import matplotlib.pyplot as plt
from data.oxford_pet import get_oxford_dataloaders, OxfordPetDataset
from data.fs2k import get_fs2k_dataloaders, FS2KDataset
from data.corruptions import CORRUPTION_NAMES

oxford_images_dir = os.path.join(RAW_OXFORD_DIR, 'images_128x128')
train_loader, val_loader, test_loader = get_oxford_dataloaders(
    manifest_dir=MANIFESTS_DIR,
    images_dir=oxford_images_dir,
    batch_size=8
)

# Fetch a training batch with runtime corruptions
batch = next(iter(train_loader))
corrupted_imgs = batch['corrupted']
clean_imgs = batch['clean']
labels = batch['label']

print(f"Batch loaded successfully! Corrupted tensor shape: {corrupted_imgs.shape}")
print(f"Corruption labels in batch: {[CORRUPTION_NAMES[l.item()] for l in labels]}")

# Display sample corrupted vs clean images
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(min(4, corrupted_imgs.shape[0])):
    corr_np = corrupted_imgs[i].permute(1, 2, 0).numpy()
    clean_np = clean_imgs[i].permute(1, 2, 0).numpy()
    
    axes[0, i].imshow(corr_np)
    axes[0, i].set_title(f"Corrupted: {CORRUPTION_NAMES[labels[i].item()]}")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(clean_np)
    axes[1, i].set_title("Clean Ground Truth")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### Step 7: Configure MLflow Persistent Tracking

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")
mlflow.set_experiment("Task1-Universal-Restoration")

with mlflow.start_run(run_name="colab_bootstrap_verification") as run:
    mlflow.log_param("device", "cuda" if torch.cuda.is_available() else "cpu")
    mlflow.log_metric("bootstrap_status", 1.0)

print(f"✅ MLflow tracking active at {MLRUNS_DIR}")
print("🎉 Environment fully verified and ready for model training!")